In [15]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

In [16]:
from sklearn.manifold import TSNE
import umap
from torchvision import datasets, transforms
from sklearn.datasets import load_iris
from numpy import reshape
import seaborn as sns
import pandas as pd
import torch
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.metrics import normalized_mutual_info_score, silhouette_score
from sklearn.cluster import KMeans

In [17]:
## Preparation for similarity method
torch.cuda.is_available()

True

# MNIST dataset TSNE fitting and visualizing

## 1. Data prepocessing

In [18]:
transform = transforms.Compose([
    transforms.ToTensor()
])

mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

n=40000

x_train = torch.cat((mnist_train.data.float(), mnist_test.data.float()),0).to(torch.float32)[:n,]
y_train = torch.cat((mnist_train.targets, mnist_test.targets),0).to(torch.float32)[:n,]

del mnist_train
del mnist_test

x_train = x_train.view(x_train.shape[0], -1)

print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")

x_train shape: torch.Size([40000, 784])
y_train shape: torch.Size([40000])


In [19]:
x_new = x_train.T
del x_train
print(x_new.shape)
type(x_new)

torch.Size([784, 40000])


torch.Tensor

## 2. Function defined 

#### Y Approximation
*   n = 70000
*   k = 10
*   ny = 500

In [20]:
def gaussian_kernel(x, y, sigma=1000.0):
    beta = 1.0 / (2.0 * sigma ** 2)
    dist = torch.cdist(x.t(), y.t()) ** 2
    # print(dist.mean())
    kernel = torch.exp(-beta * dist)
    return kernel

def mmd_loss(x, y, sigma_list=[10.0, 20.0, 30.0, 50.0]):

    mmd_list = []
    for sigma in sigma_list:
      xx_kernel = gaussian_kernel(x, x, sigma)
      yy_kernel = gaussian_kernel(y, y, sigma)
      xy_kernel = gaussian_kernel(x, y, sigma)
      mmd = xx_kernel.mean() + yy_kernel.mean() - 2 * xy_kernel.mean()
      mmd_list.append(mmd)
    return max(mmd_list)

# 假设我们有 k 个数据集 X1 到 Xk
k = 10
n = x_new.shape[1]//k
m = x_new.shape[0]
ny = 500
batch_size = 256

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
x_tensor = x_new/255
X = [x_tensor[:, i*n:(i+1)*n].to(device) for i in range(k)]

def approximate_Y(beta = 0.0):
  
  Y = torch.rand((m, ny), dtype=torch.float32, requires_grad=True, device=device)
  # print(Y)
  learning_rate = 1e-1
  # optimizer = torch.optim.SGD([Y], lr=learning_rate)
  optimizer = torch.optim.Adam([Y], lr=learning_rate)
  # optimizer = torch.optim.RMSprop([Y], lr=learning_rate)

  # 训练循环
  num_epochs = 500
  for epoch in range(num_epochs):

      total_loss = 0.0
      optimizer.zero_grad()
      total_grad = None

      for i in range(k):
          Xi = X[i]

          # mini_batch
          dataset = TensorDataset(Xi.t())
          data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

          for batch in data_loader:
            Xi_batch = batch[0].t()
            loss = mmd_loss(Xi_batch, Y)
            loss.backward()
            total_loss += loss.item()

            gradients = Y.grad
            std_grad = torch.std(gradients)
            noise = torch.randn_like(gradients) * std_grad * beta
            noisy_gradients = gradients + noise

            if total_grad is None:
              total_grad = noisy_gradients
            else:
              total_grad += noisy_gradients
            break # only use first mini-batch

          Y.grad.zero_()

      avg_grad = total_grad / k
      total_loss = total_loss / k

      with torch.no_grad():
          Y.grad = avg_grad
          Y.data=torch.clamp(Y.data, min=0, max=1)
          optimizer.step()
      if epoch== num_epochs-1:
          print('Epoch finished')
  return Y

cuda


In [21]:
import matplotlib.pyplot as plt

def visualize_Y(Y):

    # figure
    for i in range(24):
        vector = Y[:, torch.randint(0,ny,(1,1))].cpu().detach().numpy()
        image = vector.reshape(28, 28)
        plt.subplot(3, 8, i+1)
        plt.imshow(image, cmap='gray', vmin=0, vmax=1)
        plt.title('28x28 Image')
    plt.show()

    # Y tsne visualization
    Y_numpy = Y.detach().cpu().numpy()
    # tsne = TSNE(n_components=2, verbose=1)
    # z = tsne.fit_transform(Y_numpy.T)
    reducer = umap.UMAP(n_components=2)
    z = reducer.fit_transform(Y_numpy.T)

    df = pd.DataFrame()
    df["y"] = 1
    df["comp-1"] = z[:,0]
    df["comp-2"] = z[:,1]

    sns.scatterplot(x="comp-1", y="comp-2",
                    data=df).set(title="Y UMAP projection")

#### TSNE result

In [22]:
def calculate_overlap_ratio(high_dim_data, low_dim_data, k):
    """
    high_dim_data: torch data (40000*784)
    low_dim_data: z(40000*2)
    """
    high_dim_data = high_dim_data.cpu().detach().numpy()

    # 在高维空间中找到每个点的 k 近邻
    nn_high = NearestNeighbors(n_neighbors=k+1).fit(high_dim_data)
    distances_high, indices_high = nn_high.kneighbors(high_dim_data)
    
    # 在低维空间中找到每个点的 k 近邻
    nn_low = NearestNeighbors(n_neighbors=k+1).fit(low_dim_data)
    distances_low, indices_low = nn_low.kneighbors(low_dim_data)
    
    overlap_ratios = []
    
    # 对于每个点，计算两个集合的重叠比例
    for i in range(len(high_dim_data)):
        A = set(indices_high[i][1:])  # 跳过自身
        B = set(indices_low[i][1:])   # 跳过自身
        overlap = len(A.intersection(B)) / k
        overlap_ratios.append(overlap)
    
    return np.mean(overlap_ratios)

In [23]:
def tsne_res(high_dim_data, distance, y_train=y_train, visualize=False):
  tsne = TSNE(n_components=2, verbose=0, metric="precomputed", init='random')
  z = tsne.fit_transform(distance)
  # reducer = umap.UMAP(n_components=2, metric='precomputed', init='random')
  # z = reducer.fit_transform(distance)

  if visualize == True:
    df = pd.DataFrame()
    df["y"] = y_train
    df["comp-1"] = z[:,0]
    df["comp-2"] = z[:,1]

    sns.scatterplot(x="comp-1", y="comp-2", hue=df.y.tolist(),
                    palette=sns.color_palette("hls", 10),
                    data=df).set(title="MNIST data UMAP projection")

#-------------------------------------------------------------------------
  X_tr, X_te, y_tr, y_te = train_test_split(z, y_train, test_size=0.3)
  k_values = [1, 10, 50]

  knn_accuracies = {}

  result = [0, 0, 0, 0, 0, 0, 0, 0]
  # [1-nn, 10-nn, 50-nn, NMI, SC, overlap_1nn, overlap_10nn, overlap_50nn]

  for i in range(0, len(k_values)):
      # k-NN分类器
      k = k_values[i]
      knn = KNeighborsClassifier(n_neighbors=k)
      knn.fit(X_tr, y_tr)
      accuracy = knn.score(X_te, y_te)
      result[i] = accuracy

  # for key,value in knn_accuracies.items():
  #     print(f'{key}-nn:/ {value}')

  # 使用kmeans聚类
  kmeans = KMeans(n_clusters=10, n_init=10)
  y_pred = kmeans.fit_predict(z)

  # 计算NMI
  nmi = normalized_mutual_info_score(y_train, y_pred)
  result[3] = nmi
  # print('NMI:', nmi)

  # 计算轮廓系数
  sc = silhouette_score(z, y_pred)
  result[4] = sc
  # print('SC:', sc)

  # 计算overlap
  result[5] = calculate_overlap_ratio(high_dim_data, z, k=1)
  result[6] = calculate_overlap_ratio(high_dim_data, z, k=10)
  result[7] = calculate_overlap_ratio(high_dim_data, z, k=50)

  return result

#### approxiamted TSNE
- estimate x dist from dist

In [24]:
def estimate_x_dist(shuffled_X_cat, Y):
  '''
  shuffled_X_cat
  Y: learned dist
  '''
  B = torch.cdist(Y.t(), shuffled_X_cat.t())
  A = torch.cdist(Y.t(), Y.t())
  # A = A + torch.eye(A.shape[0], device = device)*1e-5
  # B = Y.t() @ X_cat
  # A = Y.t() @ Y
  res = B.t() @ torch.linalg.inv(A) @ B
  res[torch.arange(res.size(0)), torch.arange(res.size(1))] = 0
  return res

## 3. Experiments

In [25]:
X_cat = torch.cat(X, dim=1)
#-----------------
cols = X_cat.size(1)

### correct TSNE

In [26]:
# correct tsne
print('<correct TSNE>')
print('------------------')


n = 10
result = []
for i in range(0, n):

    perm = torch.randperm(cols)
    shuffled_X_cat = X_cat[:, perm]

    distance = torch.cdist(shuffled_X_cat.t(), shuffled_X_cat.t()).detach().cpu().numpy()
    result.append(tsne_res(shuffled_X_cat.t(), distance, y_train[perm]))

final_mean = np.mean(result, 0)
final_var = np.var(result, 0)**(1/2)

k_values = [1, 10, 50]

# mean
print('<mean>')
for i in range(0, 3):
    print(f'{k_values[i]}-nn:/ {final_mean[i]}')
print('NMI:', final_mean[3])
print('SC:', final_mean[4])
print('overlap_1nn:', final_mean[5])
print('overlap_10nn', final_mean[6])
print('overlap_50nn', final_mean[7])

# std
print('<std>')
for i in range(0, 3):
    print(f'{k_values[i]}-nn:/ {final_var[i]}')
print('NMI:', final_var[3])
print('SC:', final_var[4])
print('overlap_1nn:', final_var[5])
print('overlap_10nn', final_var[6])
print('overlap_50nn', final_var[7])

<correct TSNE>
------------------
<mean>
1-nn:/ 0.9617749999999999
10-nn:/ 0.96555
50-nn:/ 0.9608916666666666
NMI: 0.7747370408919848
SC: 0.4226216286420822
overlap_1nn: 0.417605
overlap_10nn 0.39053875
overlap_50nn 0.34405215000000006
<std>
1-nn:/ 0.0015152236725242054
10-nn:/ 0.0017184456801294587
50-nn:/ 0.001488404253629464
NMI: 0.024343042071764465
SC: 0.008243462045052514
overlap_1nn: 0.0015824348327814373
overlap_10nn 0.0005110140042112281
overlap_50nn 0.0006676934195422084


### approximate with beta = 0.0 (average 5 experiments)

In [27]:
print('<Approximate TSNE with beta= 0.0>')
print('------------------')

n = 10
result = []
for i in range(0, n):

    perm = torch.randperm(cols)
    shuffled_X_cat = X_cat[:, perm]

    Y = approximate_Y(beta = 0.0)
    distance = estimate_x_dist(shuffled_X_cat, Y).cpu().detach().numpy()
    result.append(tsne_res(shuffled_X_cat.t(), distance, y_train[perm]))

final_mean = np.mean(result, 0)
final_var = np.var(result, 0)**(1/2)

k_values = [1, 10, 50]

# mean
print('<mean>')
for i in range(0, 3):
    print(f'{k_values[i]}-nn:/ {final_mean[i]}')
print('NMI:', final_mean[3])
print('SC:', final_mean[4])
print('overlap_1nn:', final_mean[5])
print('overlap_10nn', final_mean[6])
print('overlap_50nn', final_mean[7])

# std
print('<std>')
for i in range(0, 3):
    print(f'{k_values[i]}-nn:/ {final_var[i]}')
print('NMI:', final_var[3])
print('SC:', final_var[4])
print('overlap_1nn:', final_var[5])
print('overlap_10nn', final_var[6])
print('overlap_50nn', final_var[7])

<Approximate TSNE with beta= 0.0>
------------------
Epoch finished
Epoch finished
Epoch finished
Epoch finished
Epoch finished
Epoch finished
Epoch finished
Epoch finished
Epoch finished
Epoch finished
<mean>
1-nn:/ 0.9400499999999999
10-nn:/ 0.9476666666666667
50-nn:/ 0.9401416666666668
NMI: 0.7534086003990317
SC: 0.44065276682376864
overlap_1nn: 0.27275750000000004
overlap_10nn 0.33728774999999994
overlap_50nn 0.33005575
<std>
1-nn:/ 0.0017220788470785873
10-nn:/ 0.001734054337223722
50-nn:/ 0.0022453810614483974
NMI: 0.020243921358309814
SC: 0.010305848707925096
overlap_1nn: 0.002240006975435572
overlap_10nn 0.0006912465641867674
overlap_50nn 0.0006803166266526256


In [28]:
print('<Approximate TSNE with beta= 1.0>')
print('------------------')

n = 10
result = []
for i in range(0, n):
    perm = torch.randperm(cols)
    shuffled_X_cat = X_cat[:, perm]

    Y = approximate_Y(beta = 1.0)
    distance = estimate_x_dist(shuffled_X_cat, Y).cpu().detach().numpy()
    result.append(tsne_res(shuffled_X_cat.t(), distance, y_train[perm]))

final_mean = np.mean(result, 0)
final_var = np.var(result, 0)**(1/2)

k_values = [1, 10, 50]

# mean
print('<mean>')
for i in range(0, 3):
    print(f'{k_values[i]}-nn:/ {final_mean[i]}')
print('NMI:', final_mean[3])
print('SC:', final_mean[4])
print('overlap_1nn:', final_mean[5])
print('overlap_10nn', final_mean[6])
print('overlap_50nn', final_mean[7])

# std
print('<std>')
for i in range(0, 3):
    print(f'{k_values[i]}-nn:/ {final_var[i]}')
print('NMI:', final_var[3])
print('SC:', final_var[4])
print('overlap_1nn:', final_var[5])
print('overlap_10nn', final_var[6])
print('overlap_50nn', final_var[7])

<Approximate TSNE with beta= 1.0>
------------------


Epoch finished
Epoch finished
Epoch finished
Epoch finished
Epoch finished
Epoch finished
Epoch finished
Epoch finished
Epoch finished
Epoch finished
<mean>
1-nn:/ 0.936375
10-nn:/ 0.9442833333333333
50-nn:/ 0.9353583333333333
NMI: 0.7470868719520543
SC: 0.4477608770132065
overlap_1nn: 0.25434750000000006
overlap_10nn 0.32634474999999996
overlap_50nn 0.32582175
<std>
1-nn:/ 0.0019522956572541343
10-nn:/ 0.0011548208134213298
50-nn:/ 0.0021848754096184704
NMI: 0.0072797452354834745
SC: 0.006557980548155653
overlap_1nn: 0.001552032941016394
overlap_10nn 0.00045153827357157807
overlap_50nn 0.0006541754065233722
